In [ ]:
from algbench import read_as_pandas
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

In [ ]:
def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = (
        AnnotatedSolution.model_validate_json(row["result"]["solution"])
        if row["result"]["solution"]
        else None
    )

    if solution is None:
        print(
            row["result"]["error"],
            row["parameters"]["args"]["kwargs"]["instance_name"],
            row["parameters"]["args"]["alg_params"],
        )
        return None

    # if row["parameters"]["args"]["alg_params"]["root"] == "LongestPair":
    #    return None

    instance_name = row["parameters"]["args"]["kwargs"]["instance_name"]

    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "is_optimal": solution.is_optimal,
        "relative_gap": solution.relative_gap,
        "relative_gap_readable": str(round((solution.alg_relative_gap - 1) * 100, 2)) + "%",
        "instance_name": instance_name,
        "instance": instance,
        "instance_type": instance.derive_instance_type(),
        "solve_time": row["result"].get("solve_time"),
        "n": instance.num_polygons(),
        "eps": row["parameters"]["args"]["alg_params"]["eps"],
        "root": row["parameters"]["args"]["alg_params"]["root"],
    }


benchmark = read_as_pandas("results_optimality_gaps_300", read_row)
print("loaded", len(benchmark), "runs")

In [ ]:
benchmark.groupby("root")["is_optimal"].value_counts()

In [ ]:
benchmark[~benchmark["is_optimal"]]["instance_name"]

In [ ]:
benchmark[benchmark["root"] == "OrderRoot"][["instance_name", "relative_gap_readable"]]